# Model Architecture

Two paradigms for MRNet: 2D slice-aggregation CNN (original) vs 3D CNN.

In [1]:
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import torchvision
import torchvision.models as tvm
import torchvision.models.video  # noqa: F401  (registers tvm.video)

SEED = 42
IMG_DIR = 'images'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

np.random.seed(SEED)
torch.manual_seed(SEED)
print('device:', DEVICE, '| torch', torch.__version__, '| torchvision', torchvision.__version__)

device: cuda | torch 2.11.0+cu128 | torchvision 0.26.0+cu128


**2D paradigm (original MRNet):** slices pass independently through a pretrained backbone; max-pool + logsumexp-pool across slices feed a logistic head.

In [2]:
class MRNet2D(nn.Module):
    def __init__(self, backbone='resnet18', pretrained=True, agg='paper', freeze_backbone=False):
        super().__init__()
        self.backbone, self.feat_dim = self._make_backbone(backbone, pretrained)
        self.agg = agg
        self.out_dim = (2 if agg == 'paper' else 1) * self.feat_dim
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.fc = nn.Linear(self.out_dim, 1)

    def _make_backbone(self, name, pretrained):
        wp = tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        wa = tvm.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None
        if name == 'resnet18':
            m = tvm.resnet18(weights=wp)
            m.fc = nn.Identity()
            return nn.Sequential(*list(m.children())), 512
        if name == 'alexnet':
            m = tvm.alexnet(weights=wa)
            return nn.Sequential(m.features, nn.AdaptiveAvgPool2d(1), nn.Flatten()), 256
        raise ValueError(name)

    def forward(self, x):                    # (B, 3, D, H, W)
        b, c, d, h, w = x.shape
        f = self.backbone(x.permute(0, 2, 1, 3, 4).reshape(b * d, c, h, w))  # (B*D, feat)
        f = f.view(b, d, self.feat_dim)
        if self.agg == 'paper':
            pooled = torch.cat([f.max(1).values, f.logsumexp(1)], dim=1)
        elif self.agg == 'max':
            pooled = f.max(1).values
        elif self.agg == 'avg':
            pooled = f.mean(1)
        else:
            raise ValueError(self.agg)
        return self.fc(pooled)

In [3]:
model2d = MRNet2D(backbone='resnet18', pretrained=True, agg='paper')
x0 = torch.randn(2, 3, 32, 256, 256)
with torch.no_grad():
    y0 = model2d(x0)
print('output:', tuple(y0.shape), '| params_M:', round(sum(p.numel() for p in model2d.parameters()) / 1e6, 2))

frozen = MRNet2D(pretrained=True, freeze_backbone=True)
print('trainable params (frozen backbone):', sum(p.numel() for p in frozen.parameters() if p.requires_grad))

output: (2, 1) | params_M: 11.18
trainable params (frozen backbone): 1025


Aggregation variants: `paper` (default: max + logsumexp concat), `max`, `avg`.

In [4]:
logits = {}
for agg in ('paper', 'max', 'avg'):
    m = MRNet2D(pretrained=False, agg=agg)
    with torch.no_grad():
        logits[agg] = m(x0).flatten().tolist()
print({k: [round(v, 3) for v in vs] for k, vs in logits.items()})

{'paper': [-2.323, -2.374], 'max': [-1.56, -1.617], 'avg': [-0.644, -0.631]}


**3D paradigm (alternative):** whole volume through a 3D CNN — Kinetics-pretrained `r3d_18` plus a lightweight from-scratch option.

In [5]:
class MRNet3D(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        m = tvm.video.r3d_18(weights=tvm.video.R3D_18_Weights.KINETICS400_V1 if pretrained else None)
        m.fc = nn.Identity()
        self.backbone = m
        self.fc = nn.Linear(512, 1)

    def forward(self, x):                # (B, 3, D, H, W)
        return self.fc(self.backbone(x).flatten(1))


class BasicBlock3D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(cout)
        self.short = nn.Sequential()
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv3d(cin, cout, 1, stride=stride, bias=False),
                                       nn.BatchNorm3d(cout))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.short(x))


class ResNet3DScratch(nn.Module):
    def __init__(self, width=16):
        super().__init__()
        w0, w1, w2 = width, width * 2, width * 4
        self.stem = nn.Sequential(nn.Conv3d(3, w0, 5, padding=2, bias=False),
                                  nn.BatchNorm3d(w0), nn.ReLU())
        self.layer1 = BasicBlock3D(w0, w0)
        self.layer2 = BasicBlock3D(w0, w1, stride=2)
        self.layer3 = BasicBlock3D(w1, w2, stride=2)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(w2, 1)

    def forward(self, x):
        x = self.layer3(self.layer2(self.layer1(self.stem(x))))
        return self.fc(self.pool(x).flatten(1))

In [6]:
model3d = MRNet3D(pretrained=True).to(DEVICE)
x3 = torch.randn(1, 3, 32, 112, 112, device=DEVICE)
with torch.no_grad():
    y3 = model3d(x3)
print('r3d_18 output:', tuple(y3.shape), '| params_M:', round(sum(p.numel() for p in model3d.parameters()) / 1e6, 2))

scratch = ResNet3DScratch(width=16)
with torch.no_grad():
    ys = scratch(x3.cpu())
print('scratch 3D output:', tuple(ys.shape), '| params_M:', round(sum(p.numel() for p in scratch.parameters()) / 1e6, 2))

r3d_18 output: (1, 1) | params_M: 33.17
scratch 3D output: (1, 1) | params_M: 0.23


Real-data integration: batch from `images/valid/sagittal` (same preprocessing as `Data_processing.ipynb`).

In [7]:
def standardize(arr):
    arr = arr.astype(np.float32)
    return (arr - arr.mean()) / (arr.std() + 1e-8)


def load_volume(path, slices=32):
    arr = standardize(np.load(path))
    arr = F.interpolate(torch.from_numpy(np.ascontiguousarray(arr))[None, None],
                        size=(slices, arr.shape[1], arr.shape[2]), mode='trilinear',
                        align_corners=False)[0, 0].numpy()
    return np.stack([arr] * 3)


d = os.path.join(IMG_DIR, 'valid', 'sagittal')
files = sorted(os.listdir(d))[40:44]
xb = torch.stack([torch.from_numpy(load_volume(os.path.join(d, f))) for f in files]).to(DEVICE)
print('real batch:', tuple(xb.shape), '| exam ids:', [f[:-4] for f in files])

real batch: (4, 3, 32, 256, 256) | exam ids: ['1170', '1171', '1172', '1173']


In [8]:
model2d = model2d.to(DEVICE)
t0 = time.time()
with torch.no_grad():
    yb = model2d(xb)
print('2D logits:', [round(v, 3) for v in yb.flatten().tolist()], f'| {time.time() - t0:.2f}s/batch')

2D logits: [1.673, 1.79, 1.678, 2.071] | 0.37s/batch


In [9]:
x112 = F.interpolate(xb, size=(32, 112, 112), mode='trilinear', align_corners=False)
t0 = time.time()
with torch.no_grad():
    yb3 = model3d(x112)
print('3D logits:', [round(v, 3) for v in yb3.flatten().tolist()], f'| {time.time() - t0:.2f}s/batch @112x112')

3D logits: [0.119, 0.249, 0.279, 0.299] | 0.14s/batch @112x112


In [10]:
rows = []
for name, m in (('2D resnet18 + paper agg', model2d), ('2D resnet18 frozen backbone', frozen),
                ('2D alexnet backbone', MRNet2D(backbone='alexnet', pretrained=False)),
                ('3D r3d_18 (Kinetics)', model3d.cpu()),
                ('3D scratch (w=16)', scratch)):
    rows.append({'model': name,
                 'params_M': round(sum(p.numel() for p in m.parameters()) / 1e6, 2),
                 'trainable_M': round(sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6, 2)})
print(pd.DataFrame(rows).to_string(index=False))

                      model  params_M  trainable_M
    2D resnet18 + paper agg     11.18        11.18
2D resnet18 frozen backbone     11.18         0.00
        2D alexnet backbone      2.47         2.47
       3D r3d_18 (Kinetics)     33.17        33.17
          3D scratch (w=16)      0.23         0.23


Recommended start: pretrained 2D backbone + `paper` aggregation; 3D models as ablation (data-hungry for 1130 exams).